In [ ]:
# Final Insights and Conclusions
print("\n" + "="*80)
print("KEY INSIGHTS & CONCLUSIONS")
print("="*80)

print("\n1. MARKET REACTION PATTERNS:")
pos_reactions = (event_analysis['PostEventReturn'] > 0).sum()
print(f"   - {pos_reactions/len(event_analysis)*100:.1f}% of earnings announcements resulted in positive returns")
print(f"   - Average post-event return: {event_analysis['PostEventReturn'].mean():.2f}%")

print("\n2. EARNINGS SURPRISE IMPACT:")
print(f"   - Correlation between surprise and returns: {correlation:.3f}")
if abs(correlation) < 0.3:
    print("   - WEAK correlation suggests market doesn't immediately react to surprises")
elif abs(correlation) < 0.6:
    print("   - MODERATE correlation suggests some market efficiency")
else:
    print("   - STRONG correlation suggests significant market reaction to surprises")

print("\n3. VOLATILITY INSIGHTS:")
vol_increase = event_analysis['PostEventVolatility'].mean() - event_analysis['EstimationVolatility'].mean()
print(f"   - Volatility increases by {vol_increase:.2f}% post-earnings")
print(f"   - Average volume spike ratio: {event_analysis['VolumeSpikeRatio'].mean():.2f}x")

print("\n4. MODEL PERFORMANCE:")
print(f"   - Best model: {best_model_name}")
best_acc = model_comparison.loc[model_comparison['Model'] == best_model_name, 'Accuracy'].values[0]
best_f1 = model_comparison.loc[model_comparison['Model'] == best_model_name, 'F1-Score'].values[0]
print(f"   - Accuracy: {best_acc:.4f}")
print(f"   - F1-Score: {best_f1:.4f}")

print("\n5. PREDICTABILITY:")
if best_f1 > 0.6:
    print("   - Models show GOOD predictive power for market reactions")
elif best_f1 > 0.5:
    print("   - Models show MODERATE predictive power")
else:
    print("   - Models show LIMITED predictive power - market reactions are largely unpredictable")

print("\n6. TOP PREDICTIVE FEATURES:")
for i, row in importance_df.head(5).iterrows():
    print(f"   - {row['Feature']}: {row['Importance']:.4f}")

print("\n7. INVESTMENT IMPLICATIONS:")
print("   - Consider earnings surprise as a trading signal")
print("   - Monitor volume spikes as event indicators")
print("   - Use ensemble models for better prediction accuracy")
print("   - Sector analysis may provide additional insights")
print("   - Historical patterns may not guarantee future performance")

print("\n" + "="*80)

In [ ]:
# Stock-wise performance summary
print("\n" + "="*80)
print("STOCK-WISE PERFORMANCE SUMMARY")
print("="*80)

for ticker in event_analysis['Ticker'].unique():
    ticker_data = event_analysis[event_analysis['Ticker'] == ticker]
    
    print(f"\n{ticker}:")
    print(f"  Events analyzed: {len(ticker_data)}")
    print(f"  Avg post-event return: {ticker_data['PostEventReturn'].mean():.2f}%")
    print(f"  Positive reactions: {(ticker_data['PostEventReturn'] > 0).sum()} ({(ticker_data['PostEventReturn'] > 0).sum() / len(ticker_data) * 100:.1f}%)")
    print(f"  Avg CAR: {ticker_data['AbnormalReturnCAR'].mean():.2f}%")
    print(f"  Avg volatility increase: {(ticker_data['PostEventVolatility'].mean() - ticker_data['EstimationVolatility'].mean()):.2f}%")
    print(f"  Avg volume spike ratio: {ticker_data['VolumeSpikeRatio'].mean():.2f}x")

In [ ]:
# Earnings Surprise vs Market Reaction
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['green' if x > 0 else 'red' for x in event_analysis['PostEventReturn']]

scatter = ax.scatter(
    event_analysis['SurprisePercentage'],
    event_analysis['PostEventReturn'],
    c=colors, alpha=0.6, s=100, edgecolors='black', linewidth=0.5
)

# Add trend line
valid_mask = event_analysis['SurprisePercentage'].notna() & event_analysis['PostEventReturn'].notna()
if valid_mask.sum() > 1:
    z = np.polyfit(
        event_analysis[valid_mask]['SurprisePercentage'],
        event_analysis[valid_mask]['PostEventReturn'],
        1
    )
    p = np.poly1d(z)
    x_range = np.linspace(
        event_analysis['SurprisePercentage'].min(),
        event_analysis['SurprisePercentage'].max(),
        100
    )
    ax.plot(x_range, p(x_range), "b--", alpha=0.8, linewidth=2, label='Trend')

correlation = event_analysis['SurprisePercentage'].corr(event_analysis['PostEventReturn'])

ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('Earnings Surprise (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Post-Event Return (%)', fontsize=12, fontweight='bold')
ax.set_title(f'Earnings Surprise vs Market Reaction\nCorrelation: {correlation:.3f}', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nEarnings Surprise Analysis:")
print(f"  Correlation (Surprise vs Post-Event Return): {correlation:.3f}")
print(f"  Positive surprise events: {(event_analysis['SurprisePercentage'] > 0).sum()}")
print(f"  Negative surprise events: {(event_analysis['SurprisePercentage'] < 0).sum()}")

In [ ]:
# Feature importance analysis
best_model = model_trainer.models[best_model_name]
importance_df = feature_engineer.get_feature_importance_df(best_model, feature_columns)

print(f"\nTop 10 Important Features ({best_model_name}):")
print(importance_df.head(10))

# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 8))
top_features = importance_df.head(15)

ax.barh(range(len(top_features)), top_features['Importance'].values, color='steelblue', edgecolor='black')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.set_xlabel('Importance Score', fontweight='bold')
ax.set_title(f'Top 15 Feature Importances - {best_model_name}', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Section 11: Visualization and Insights

Create comprehensive visualizations including earnings surprise vs price reaction, feature importance, and final insights.

In [ ]:
# Confusion matrices for each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (model_name, model) in enumerate(model_trainer.models.items()):
    cm = model_trainer.results[model_name]['ConfusionMatrix']
    
    im = axes[idx].imshow(cm, cmap='Blues', aspect='auto')
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    
    # Add text annotations
    for i in range(2):
        for j in range(2):
            text = axes[idx].text(j, i, str(cm[i, j]),
                                 ha="center", va="center", color="black", fontweight='bold')
    
    axes[idx].set_xticks([0, 1])
    axes[idx].set_yticks([0, 1])
    axes[idx].set_xticklabels(['Negative', 'Positive'])
    axes[idx].set_yticklabels(['Negative', 'Positive'])

plt.tight_layout()
plt.show()

In [ ]:
# Model evaluation visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metrics_df = model_comparison.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
metrics_df.plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_xlabel('')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0, 1.1])
axes[0].tick_params(axis='x', rotation=45)

# AUC-ROC comparison
auc_data = model_comparison.set_index('Model')['AUC-ROC'].dropna()
colors = plt.cm.viridis(np.linspace(0, 1, len(auc_data)))
axes[1].bar(auc_data.index, auc_data.values, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_title('AUC-ROC Score Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_xlabel('')
axes[1].set_ylim([0, 1.1])
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=45)

# Add values on bars
for i, (idx, val) in enumerate(auc_data.items()):
    axes[1].text(i, val, f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Best model
best_model_name = model_comparison.loc[model_comparison['F1-Score'].idxmax(), 'Model']
print(f"\nBest Model: {best_model_name}")
print(f"  F1-Score: {model_comparison.loc[model_comparison['Model'] == best_model_name, 'F1-Score'].values[0]:.4f}")

## Section 10: Model Evaluation and Comparison

Evaluate models using accuracy, precision, recall, and F1-score metrics.

In [ ]:
# Prepare training data
feature_columns = [col for col in features.columns if col not in 
                  ['Ticker', 'EarningsDate', 'PostEventReturn', 'PreEventReturn', 
                   'EPS', 'SurprisePercentage', 'DataPointsInWindow', 'DataPointsInEstimation']]

X_train, X_test, y_train, y_test = feature_engineer.prepare_train_test_data(
    features,
    labels,
    feature_columns=feature_columns,
    test_size=0.2,
    random_state=42
)

print(f"Training Data Preparation:")
print(f"  Features used: {len(feature_columns)}")
print(f"  Training set size: {len(X_train)}")
print(f"  Test set size: {len(X_test)}")
print(f"  Training set label distribution:")
print(f"    Positive: {(y_train == 1).sum()} ({(y_train == 1).sum() / len(y_train) * 100:.1f}%)")
print(f"    Negative: {(y_train == 0).sum()} ({(y_train == 0).sum() / len(y_train) * 100:.1f}%)")

# Train models
print("\nTraining Models...")
model_trainer = ModelTrainer()
model_comparison = model_trainer.train_all_models(X_train, X_test, y_train, y_test)

print("\nModel Comparison:")
print(model_comparison)

## Section 9: Train Classification Models

Train Logistic Regression, Random Forest, and XGBoost models.

In [ ]:
# Create labels
LABEL_THRESHOLD = 0.0
labels = feature_engineer.create_binary_labels(event_analysis, threshold=LABEL_THRESHOLD)

print(f"Label Distribution (threshold: {LABEL_THRESHOLD}%):")
print(f"  Positive reactions (1): {(labels == 1).sum()} ({(labels == 1).sum() / len(labels) * 100:.1f}%)")
print(f"  Negative reactions (0): {(labels == 0).sum()} ({(labels == 0).sum() / len(labels) * 100:.1f}%)")

# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label counts
label_counts = pd.Series(labels).value_counts()
axes[0].bar(['Negative', 'Positive'], [label_counts[0], label_counts[1]], color=['red', 'green'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Label Distribution')
axes[0].grid(True, alpha=0.3, axis='y')

# Add percentages
for i, (label, count) in enumerate([(0, label_counts[0]), (1, label_counts[1])]):
    axes[0].text(i, count, f'{count/len(labels)*100:.1f}%', ha='center', va='bottom', fontweight='bold')

# Returns by label
axes[1].boxplot([event_analysis[labels == 0]['PostEventReturn'].values,
                 event_analysis[labels == 1]['PostEventReturn'].values],
                labels=['Negative', 'Positive'])
axes[1].set_ylabel('Post-Event Return (%)')
axes[1].set_title('Return Distribution by Label')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section 8: Data Labeling

Create target variable for classification models (positive/negative market reaction).

In [ ]:
# Feature Engineering
feature_engineer = FeatureEngineer()
features = feature_engineer.create_features(event_analysis)

print("Feature Engineering Results:")
print(f"  Original features: {len(event_analysis.columns)}")
print(f"  Engineered features: {len(features.columns)}")
print(f"  New features created: {len(features.columns) - len(event_analysis.columns)}")

print(f"\nFeature Columns:")
feature_cols = [col for col in features.columns if col not in event_analysis.columns]
for i, col in enumerate(feature_cols, 1):
    print(f"  {i}. {col}")

print(f"\nFeature Statistics:")
numeric_features = features.select_dtypes(include=[np.number]).columns
print(features[numeric_features].describe())

## Section 7: Feature Engineering

Generate features for machine learning models including earnings surprise, volatility changes, and momentum indicators.

In [ ]:
# Volatility Analysis
print("Volatility Analysis:")
print(f"  Avg Estimation Volatility: {event_analysis['EstimationVolatility'].mean():.2f}%")
print(f"  Avg Pre-Event Volatility: {event_analysis['PreEventVolatility'].mean():.2f}%")
print(f"  Avg Post-Event Volatility: {event_analysis['PostEventVolatility'].mean():.2f}%")
print(f"  Volatility Increase (Post vs Estimation): {event_analysis['PostEventVolatility'].mean() - event_analysis['EstimationVolatility'].mean():.2f}%")

# Volume Analysis
print(f"\nVolume Analysis:")
print(f"  Avg Volume Spike Ratio: {event_analysis['VolumeSpikeRatio'].mean():.2f}x")
print(f"  Events with >1.5x volume increase: {(event_analysis['VolumeSpikeRatio'] > 1.5).sum()}")
print(f"  Events with >2.0x volume increase: {(event_analysis['VolumeSpikeRatio'] > 2.0).sum()}")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Volatility by event
x_pos = np.arange(min(20, len(event_analysis)))
axes[0, 0].plot(x_pos, event_analysis['EstimationVolatility'].values[:20], marker='o', label='Estimation', alpha=0.7)
axes[0, 0].plot(x_pos, event_analysis['PreEventVolatility'].values[:20], marker='s', label='Pre-Event', alpha=0.7)
axes[0, 0].plot(x_pos, event_analysis['PostEventVolatility'].values[:20], marker='^', label='Post-Event', alpha=0.7)
axes[0, 0].set_ylabel('Volatility (%)')
axes[0, 0].set_title('Volatility Around First 20 Earnings Events')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Volume spike ratio distribution
axes[0, 1].hist(event_analysis['VolumeSpikeRatio'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='No Spike')
axes[0, 1].axvline(x=event_analysis['VolumeSpikeRatio'].mean(), color='green', linestyle='--', linewidth=2, label='Average')
axes[0, 1].set_xlabel('Volume Spike Ratio')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Volume Spike Ratios')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Volatility vs Returns
axes[1, 0].scatter(event_analysis['PostEventVolatility'], event_analysis['PostEventReturn'], alpha=0.6, s=100)
axes[1, 0].set_xlabel('Post-Event Volatility (%)')
axes[1, 0].set_ylabel('Post-Event Return (%)')
axes[1, 0].set_title('Volatility vs Returns')
axes[1, 0].grid(True, alpha=0.3)

# Volume Spike Ratio vs Returns
axes[1, 1].scatter(event_analysis['VolumeSpikeRatio'], event_analysis['PostEventReturn'], alpha=0.6, s=100, color='orange')
axes[1, 1].set_xlabel('Volume Spike Ratio')
axes[1, 1].set_ylabel('Post-Event Return (%)')
axes[1, 1].set_title('Volume Spike vs Returns')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 6: Volatility and Volume Analysis

Calculate volatility changes and volume spikes around earnings events.

In [ ]:
# Analyze returns around earnings
print("Return Analysis:")
print(f"  Avg Pre-Event Return: {event_analysis['PreEventReturn'].mean():.2f}%")
print(f"  Avg Post-Event Return: {event_analysis['PostEventReturn'].mean():.2f}%")
print(f"  Avg CAR: {event_analysis['AbnormalReturnCAR'].mean():.2f}%")

print(f"\nReturn Distribution:")
print(f"  Positive post-event returns: {(event_analysis['PostEventReturn'] > 0).sum()} ({(event_analysis['PostEventReturn'] > 0).sum() / len(event_analysis) * 100:.1f}%)")
print(f"  Negative post-event returns: {(event_analysis['PostEventReturn'] < 0).sum()} ({(event_analysis['PostEventReturn'] < 0).sum() / len(event_analysis) * 100:.1f}%)")

# Visualization: Return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(event_analysis['PostEventReturn'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Post-Event Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Post-Event Returns')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(event_analysis['AbnormalReturnCAR'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('CAR (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Cumulative Abnormal Returns')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero CAR')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 5: Calculate Abnormal Returns and CAR

Compute pre/post-earnings returns and cumulative abnormal returns (CAR).

In [ ]:
# Initialize event analyzer
event_analyzer = EventAnalyzer(event_window=EVENT_WINDOW, estimation_window=ESTIMATION_WINDOW)

# Analyze events
print(f"Analyzing earnings events with ±{EVENT_WINDOW} trading day windows...")
event_analysis = event_analyzer.analyze_multiple_events(price_data, earnings_data)

print(f"\nEvent analysis shape: {event_analysis.shape}")
print(f"Columns: {list(event_analysis.columns)}")
print(f"\nFirst few events:")
print(event_analysis.head())

# Summary statistics
print(f"\nEvent Analysis Summary:")
print(f"  Total events: {len(event_analysis)}")
print(f"  Unique stocks: {event_analysis['Ticker'].nunique()}")
print(f"  Date range: {event_analysis['EarningsDate'].min()} to {event_analysis['EarningsDate'].max()}")

## Section 4: Event Window Analysis

Define event windows (±5 trading days around earnings) and extract price data for analysis.

In [ ]:
# Fetch earnings data
print("Fetching earnings announcement data...")
earnings_data_list = []

for ticker in TICKERS:
    earnings_data = collector.fetch_earnings_dates(ticker)
    if not earnings_data.empty:
        earnings_data_list.append(earnings_data)

# Combine all earnings data
if earnings_data_list:
    earnings_data = pd.concat(earnings_data_list, ignore_index=True)
    print(f"\nEarnings data shape: {earnings_data.shape}")
    print(f"Columns: {list(earnings_data.columns)}")
    print(f"\nFirst few rows:")
    print(earnings_data.head(10))
    print(f"\nEarnings summary by stock:")
    print(earnings_data['Ticker'].value_counts())
else:
    print("No earnings data collected!")

## Section 3: Collect Earnings Announcement Data

Gathering quarterly earnings announcement dates, reported EPS, and expected EPS from Yahoo Finance.

In [ ]:
# Initialize data collector
collector = DataCollector(lookback_years=LOOKBACK_YEARS)

# Fetch price data
print("Fetching historical price data...")
price_data_list = []

for ticker in TICKERS:
    price_data = collector.fetch_stock_price_data(ticker)
    if not price_data.empty:
        price_data_list.append(price_data)

# Combine all price data
if price_data_list:
    price_data = pd.concat(price_data_list, ignore_index=True)
    print(f"\nPrice data shape: {price_data.shape}")
    print(f"Columns: {list(price_data.columns)}")
    print(f"\nFirst few rows:")
    print(price_data.head())
else:
    print("No price data collected!")

## Section 2: Fetch Historical Stock Price Data

Using Yahoo Finance API (yfinance) to retrieve historical OHLCV data for selected stocks.

In [ ]:
# Configuration
LOOKBACK_YEARS = 5
EVENT_WINDOW = 5
ESTIMATION_WINDOW = 60

# Select stocks to analyze (modify as needed)
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']

print(f"Configuration:")
print(f"  Lookback years: {LOOKBACK_YEARS}")
print(f"  Event window: ±{EVENT_WINDOW} trading days")
print(f"  Stocks to analyze: {', '.join(TICKERS)}")

## Section 1: Import Required Libraries

We import all necessary libraries for data collection, analysis, and visualization:
- **yfinance**: Yahoo Finance API for stock data
- **pandas/numpy**: Data manipulation
- **scikit-learn**: ML models and evaluation
- **matplotlib/seaborn**: Visualization
- **Custom modules**: Data collection, analysis, and modeling

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import logging
import warnings

warnings.filterwarnings('ignore')

# Import custom modules
from src.data_collection import DataCollector
from src.event_analysis import EventAnalyzer
from src.feature_engineering import FeatureEngineer
from src.ml_models import ModelTrainer
from src.visualizations import Visualizer

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("Libraries imported successfully!")

# Earnings Impact & Market Reaction Analysis
## Interactive Analysis Notebook

This notebook provides an interactive walkthrough of the earnings announcement impact analysis system.
It covers data collection, event analysis, feature engineering, model training, and visualization.